# Jeff original notebook — recorded source revision

Use a fresh Colab **L4** GPU runtime. This runs the original notebook 83 runner from commit `8ad4afeb347c16baeecfcfd9b7640608ac2f072b`, without modifying its sampler, model loader, rollout or success check. Both historical arms run: stock with measurement-only uncertainty probes, and P&P refinement. Two shards run sequentially: 400 episodes per arm, 800 rollouts total. No Q correction.

Setup pins the recorded Torch/CUDA versions and checks out the historical source instead of the moving main branch. A fresh experiment name prevents reuse or overwrite of Jeff’s reference results. Historical model-file revisions and several environment versions were not recorded; using this original code does not recover those missing details. Results and trajectories use the original Supabase logging. Rerunning resumes the same experiment.


In [ ]:
import os, sys, subprocess
from pathlib import Path
from google.colab import userdata
for key in ('SUPABASE_URL', 'SUPABASE_SERVICE_KEY', 'HF_TOKEN'):
    try:
        value = userdata.get(key)
        if value: os.environ[key] = value
    except Exception:
        pass
for key in ('SUPABASE_URL', 'SUPABASE_SERVICE_KEY', 'HF_TOKEN'):
    if not os.environ.get(key):
        raise RuntimeError(f'Add {key} in Colab Secrets and allow notebook access.')
if any(name == 'pnp' or name.startswith('pnp.') for name in sys.modules):
    raise RuntimeError('Start a fresh runtime: another pnp source version is already imported.')
SOURCE_REVISION = '8ad4afeb347c16baeecfcfd9b7640608ac2f072b'
repo_dir = Path('/content/jeff-original-source')
if not (repo_dir / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/ArjunS07/cs159-sp26.git', str(repo_dir)], check=True)
subprocess.run(['git', '-C', str(repo_dir), 'checkout', '--detach', SOURCE_REVISION], check=True)
assert subprocess.check_output(['git', '-C', str(repo_dir), 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_REVISION
assert not subprocess.check_output(['git', '-C', str(repo_dir), 'status', '--porcelain'], text=True).strip()
package_dir = repo_dir / 'pnp-vla'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(package_dir)+'[sim]',
                'torch==2.11.0+cu128', 'torchvision==0.26.0+cu128',
                '--extra-index-url', 'https://download.pytorch.org/whl/cu128'], check=True)
sys.path.insert(0, str(package_dir))
from pnp import env_setup
env_setup.setup_environment()
import torch
if torch.__version__ != '2.11.0+cu128':
    raise RuntimeError('Restart the runtime to load the installed Torch 2.11.0+cu128.')
print({'source_revision': SOURCE_REVISION, 'gpu': torch.cuda.get_device_name(0), 'source': str(package_dir)})


In [ ]:
# Record the actual installed simulator/grader code without changing it.
import inspect, hashlib, json, importlib.metadata
from pathlib import Path
from libero.libero.envs import OffScreenRenderEnv
from pnp.store import SupabaseStore
import libero.libero.envs as libero_envs
EXPERIMENT = 'smolvla-libero-jeff-original-notebook-v1'
root = Path(libero_envs.__file__).parent
manifest = {'source_revision': SOURCE_REVISION,
            'versions': {name:importlib.metadata.version(name) for name in ('libero','mujoco','robosuite','torch','transformers')},
            'check_success_source': inspect.getsource(OffScreenRenderEnv.check_success),
            'libero_env_source_sha256': {str(p.relative_to(root)):hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(root.rglob('*.py'))}}
SupabaseStore()._upload(f'smolvla_pcp_sweep/{EXPERIMENT}/grader_source_manifest.json', json.dumps(manifest,indent=2).encode())
print({'versions':manifest['versions'], 'check_success_source':manifest['check_success_source'],
       'grader_source_manifest_saved':True})


In [ ]:
from pnp.experiments import run_smolvla_libero_worker
EXPERIMENT = 'smolvla-libero-jeff-original-notebook-v1'
# Same original runner and arguments as notebook 83; sequential shards fit one Colab runtime.
for shard_index in (0, 1):
    print(f'Running Jeff original notebook shard {shard_index + 1}/2', flush=True)
    run_smolvla_libero_worker(shard_count=2, shard_index=shard_index,
                             experiment=EXPERIMENT, rollout_batch_size=8)


In [ ]:
from pnp.store import SupabaseStore
store = SupabaseStore()
fields = 'rollout_id,suite,task_idx,episode_idx,init_state_hash,method,status,success,episode_seed,n_steps,bddl_sha256'
rows = store.fetch_all('rollouts', fields, configure=lambda q:q.eq('experiment', EXPERIMENT), order_by=('rollout_id',))
for method in sorted({r['method'] for r in rows}):
    completed = [r for r in rows if r['method']==method and r['status']=='completed']
    print({'method':method, 'completed':len(completed), 'planned':400,
           'successes':sum(bool(r['success']) for r in completed),
           'errors':sum(r['method']==method and r['status']!='completed' for r in rows)})
reference = store.fetch_all('rollouts', fields, configure=lambda q:q.eq('experiment', 'smolvla-libero-a10-pnp-k5-steps34-v1').eq('method', 'pnp_uncertainty_only').eq('status', 'completed'), order_by=('rollout_id',))
identity = lambda r:(r['suite'],r['task_idx'],r['episode_idx'],r['init_state_hash'])
old = {identity(r):r for r in reference}
new = {identity(r):r for r in rows if r['method']=='pnp_uncertainty_only' and r['status']=='completed'}
pairs = [(old[k],new[k]) for k in old.keys() & new.keys()]
print({'paired':len(pairs), 'historical_successes':sum(a['success'] for a,b in pairs),
       'new_successes':sum(b['success'] for a,b in pairs),
       'outcome_matches':sum(a['success']==b['success'] for a,b in pairs),
       'seed_mismatches':sum(a['episode_seed']!=b['episode_seed'] for a,b in pairs),
       'task_definition_mismatches':sum(a['bddl_sha256']!=b['bddl_sha256'] for a,b in pairs)})
